## Colab setup

Run this section first in Google Colab to clone the repo and make the `pqr/` package importable.

In [ ]:
# If you're running in Google Colab, set this to your GitHub repo URL.
# Example:
# REPO_URL = "https://github.com/<OWNER>/<REPO>.git"
REPO_URL = ""

# Folder name to clone into (can be any name)
REPO_DIR = "private_query_explanations"

import os
import sys
import subprocess

def _run(cmd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)

if REPO_URL:
    if not os.path.exists(REPO_DIR):
        _run(["git", "clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
else:
    print("Set REPO_URL above, then re-run this cell.")

# Make the repo importable as a package (so `import pqr` works)
sys.path.insert(0, os.path.abspath("."))

# (Optional) install extra dependencies if you extend the notebook.
# The core implementation uses only the Python standard library.
# _run([sys.executable, "-m", "pip", "install", "-q", "python-pptx"])

## Private Query Refinement — Notebook Demo

This notebook demonstrates the reference implementation in `pqr/` for the core pseudocode in `Private_Query_Refinement.pdf`:

- **Algorithm 1**: *Private-Queries-Diff-Top-K-Explanation*
- **Algorithm 2**: *Find-Top-k-Explanations*

It runs on the included toy dataset (`pqr/toy.csv`) and prints the resulting **differentially private** (noisy) histograms explaining the differences between two queries.

In [ ]:
import json
import random

from pqr.algorithm import generate_simple_predicate_views, private_queries_diff_topk_explanation
from pqr.query import Condition, Op, Query

In [ ]:
# Load the toy dataset shipped with the repo
import csv
from pathlib import Path

csv_path = Path("pqr/toy.csv")
rows = []
with csv_path.open(newline="", encoding="utf-8") as f:
    for r in csv.DictReader(f):
        rows.append(dict(r))

len(rows), rows[0]

In [ ]:
# Define two queries Q1 and Q2 (selection predicates only)
# Q1: department == 'sales'
# Q2: department == 'engineering'
q1 = Query((Condition("department", Op.EQ, "sales"),))
q2 = Query((Condition("department", Op.EQ, "engineering"),))

# Attributes we want explanations for
attrs = ["gender", "seniority"]

# Predicate-space attributes used to generate view predicates
predicate_views = generate_simple_predicate_views(rows, predicate_attrs=["country"])

In [ ]:
rng = random.Random(7)

explanations = private_queries_diff_topk_explanation(
    dataset=rows,
    q1=q1,
    q2=q2,
    attributes=attrs,
    predicate_views=predicate_views,
    tau=3,
    k=2,
    epsilon=1.0,
    rng=rng,
)

print(json.dumps(explanations, indent=2, sort_keys=True))

### Notes

- The draft paper references some subroutines without fully specifying them (e.g., the exact One-Shot Top-τ primitive and a detailed diversity penalty). This repo uses standard DP defaults for those pieces.
- Released histograms are **noisy** (Laplace mechanism), so counts can be negative in the raw output.